# 06 — Bubble wall tracking: 1+1D vs 3+1D

For each available time snapshot in both BubbleMaster (1+1D) and sledgehamr (3+1D),
finds the outer bubble wall position by locating where φ crosses the φ(R_out) threshold
from the instanton profile.  Compares wall trajectories to diagnose any systematic offset.

In [ ]:
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import BubbleMasterParams

REPO_ROOT = Path(pt.__file__).parent.parent

PYSLEDGEHAMR_PATH = '/global/cfs/cdirs/m3166/buschman/sledgehamr'
sys.path.append(PYSLEDGEHAMR_PATH)
import pySledgehamr as sledgehamr

config = pt.Config()

sh_base = '/pscratch/sd/b/buschman/other_runs/'

VALIDATION_POINTS = [
    dict(lambda_bar=0.84,  d=55.55,
         sh_path=sh_base+'bubble_N3_234_0_1_',
         sh_L=200., sh_axis='x', sh_mid_x=100.025, sh_lev=0),
    dict(lambda_bar=0.069, d=12.54,
         sh_path=sh_base+'bubble_N3_456_0_1_',
         sh_L=50.,  sh_axis='x', sh_mid_x=22.50,   sh_lev=0),
]

def vp_tag(vp):
    return f'lb{vp["lambda_bar"]}_g{vp["gamma"]:.4g}'

for vp in VALIDATION_POINTS:
    model   = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    t_coll  = float(model.kinematics.collision_time(vp['d']))
    gamma   = float(model.kinematics.Gamma(t_coll))
    profile = model.instanton
    vp['t_coll']        = t_coll
    vp['gamma']         = gamma
    vp['phi_threshold'] = float(profile.interp(profile.rout_0))
    vp['rout_0']        = profile.rout_0
    print(f"{vp_tag(vp)}  d={vp['d']}  t_coll={t_coll:.3f}  "
          f"rout_0={profile.rout_0:.3f}  phi_threshold={vp['phi_threshold']:.4f}")

## 1. BubbleMaster wall tracking

In [ ]:
def find_outer_wall(z, phi, threshold):
    """Return the outermost z where phi exceeds threshold; NaN if none."""
    idx = np.where(phi > threshold)[0]
    return float(z[idx[-1]]) if len(idx) > 0 else np.nan


bm_tracks = {}
for vp in VALIDATION_POINTS:
    tag     = vp_tag(vp)
    model   = pt.PhysicsModel(pt.Phi4Potential(vp['lambda_bar']), config)
    out_dir = model.results_dir / 'validation' / tag / 'out'
    fields_path = out_dir / 'fields.h5'
    if not fields_path.exists():
        print(f'[{tag}]  fields.h5 not found — run notebook 04 first')
        continue
    with h5py.File(fields_path, 'r') as f:
        s_bm   = f['s'][:]
        z_bm   = f['z'][:]
        phi_bm = f['phi'][:]   # shape (n_s, n_z)
    thr   = vp['phi_threshold']
    walls = np.array([find_outer_wall(z_bm, phi_bm[i, :], thr)
                      for i in range(len(s_bm))])
    # s=0 in BubbleMaster is the collision moment; shift to absolute Minkowski time
    t_abs = s_bm + vp['t_coll']
    bm_tracks[tag] = dict(t=t_abs, z_wall=walls, s=s_bm)
    valid = np.isfinite(walls)
    print(f'[{tag}]  {valid.sum()}/{len(s_bm)} snapshots with wall  '
          f's=[{s_bm[0]:.2f},{s_bm[-1]:.2f}]')

## 2. sledgehamr wall tracking

In [ ]:
sh_tracks = {}
for vp in VALIDATION_POINTS:
    tag    = vp_tag(vp)
    output = sledgehamr.Output(vp['sh_path'])
    t_snaps = output.GetTimesOfSlices()
    thr   = vp['phi_threshold']
    L     = vp['sh_L']
    mid_x = vp['sh_mid_x']
    axis  = vp['sh_axis']
    lev   = vp['sh_lev']

    t_out    = []
    z_wall_out = []
    n_fail   = 0

    for idx in range(len(t_snaps)):
        try:
            slices = output.GetSlice(idx, axis, lev, ['Phi'])
            t      = slices['t']
            phi_2d = slices['Phi']
            n_rows, n_cols = phi_2d.shape
            phi_1d = phi_2d[n_rows // 2, :]
            z_sh   = np.linspace(0, L, n_cols) - mid_x
            # only right half (z >= 0)
            mask   = z_sh >= 0
            w      = find_outer_wall(z_sh[mask], phi_1d[mask], thr)
            t_out.append(t)
            z_wall_out.append(w)
        except Exception:
            n_fail += 1

    sh_tracks[tag] = dict(t=np.array(t_out), z_wall=np.array(z_wall_out))
    valid = np.isfinite(z_wall_out)
    print(f'[{tag}]  {len(t_out)} slices loaded, {n_fail} skipped, '
          f'{np.sum(valid)} with wall found  '
          f't=[{t_out[0]:.2f},{t_out[-1]:.2f}]')

## 3. Plot

In [ ]:
colors = ['#0072B2', '#D55E00']

fig, axes = plt.subplots(1, len(VALIDATION_POINTS),
                          figsize=(5.5 * len(VALIDATION_POINTS), 4),
                          squeeze=False)

for col, (vp, color) in enumerate(zip(VALIDATION_POINTS, colors)):
    ax  = axes[0, col]
    tag = vp_tag(vp)

    bm = bm_tracks.get(tag)
    sh = sh_tracks.get(tag)

    if sh is not None:
        mask = np.isfinite(sh['z_wall'])
        ax.plot(sh['t'][mask], sh['z_wall'][mask],
                color=color, lw=1.5, label='3+1D', zorder=3)

    if bm is not None:
        mask = np.isfinite(bm['z_wall'])
        ax.plot(bm['t'][mask], bm['z_wall'][mask],
                color=color, lw=1.5, ls='--', label='1+1D  (s + t_coll)', zorder=3)

    # Reference: free relativistic expansion from d/2 at t_coll
    t_coll = vp['t_coll']
    d      = vp['d']
    t_ref  = np.linspace(t_coll, t_coll + 1.2 * d, 300)
    rout   = vp['rout_0']
    # outer wall trajectory: sqrt(rout^2 + t^2) — single free bubble
    z_ref  = np.sqrt(rout**2 + t_ref**2)
    ax.plot(t_ref, z_ref, 'k:', lw=1, label=r'free: $\sqrt{r_{out}^2+t^2}$', zorder=2)

    ax.axvline(t_coll, color='grey', lw=0.8, ls=':')
    ax.text(t_coll, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 0,
            ' $t_{coll}$', fontsize=7, va='bottom', color='grey')

    ax.set_xlabel('Minkowski time $t$')
    ax.set_ylabel('Outer wall position $z$')
    ax.set_title(rf"$\bar{{\lambda}}={vp['lambda_bar']}$", fontsize=9)
    ax.legend(frameon=False, fontsize=8)

plt.tight_layout()
plt.savefig(REPO_ROOT / 'bubble_wall_tracking.pdf', bbox_inches='tight')
plt.show()